In [0]:
select * from (
  select 
    group_id,
    groupname,
    salesforce_facility_id as facility_id,
    cust_name as customer_name,
    limit_desc as facility_description,
    risk_country_code as country,
    sales_sol_id as region,
    oau_region,
    main_dept_code_desc as department,
    trade_direct as direction_of_trade,
    primaryrmlogin_id as client_relationship_manager,
    primary_dept_rm as product_manager,
    date_format(lim_sanct_date, 'MM/dd/yyyy') as facility_approval_date,
    lim_contract_date as contract_sign_date,
    date_format(firstdisbursementdate, 'MM/dd/yyyy') as first_disbursement_date,
    originalfacilitygrade as original_facility_grade,
    loan_grade as facility_grade,
    riskcategory as risk_category,
    status_code as facility_status,
    availability_end_date as `availability_period-end_date`,
    (case when lim_exp_date > orig_lim_exp_date then date_format(lim_exp_date, 'MM/dd/yyyy') else date_format(orig_lim_exp_date, 'MM/dd/yyyy') end) as facility_expiry_date,
    sector_being_financed as sector,
    subsector as `sub-sector`,
    reportingsector as reporting_sector,
    interesttablecode as interest_table_code,
    accountmargin as account_margin_pct,
    program_loan_type as program,
    totalfacilityamountusd as total_facility_amt_usd,
    totalfacilityamtorgccy as total_facility_amt_org_ccy,
    totalfacilitycurrency as total_facility_currency,
    facilitytype as facility_type,
    legalentity_type as beneficiary_type,
    borrowertype as borrower_type,
    orig_sanct_lim as approved_limit,
    approvedlimitorgccy as approved_limit_org_ccy,
    committed,
    coalesce(undrawn_lim,0) as undrawn_limit,
    coalesce(undrawnlimitorgccy,0) as undrawn_limit_orginal_ccy,
    coalesce(funded_out_bal,0) as principal_bal_funded,
    coalesce(total_exp_usd,0) as total_exposure_funded,
    coalesce(funded_out_bal_org,0) as principal_bal_funded_org_ccy,
    coalesce(total_exp,0) as total_exposure_funded_org_ccy,
    coalesce(cont_out_bal,0) as outstanding_bal_contig,
    coalesce(cont_out_bal_org,0) as outstand_bal_contig_org_ccy,
    coalesce((coalesce(undrawn_lim,0) + coalesce(total_exp_usd,0) + coalesce(cont_out_bal,0)),0) as gross_exposure,
    coalesce(case when status_code = 'NON-OPERATIONAL' then 0 else (case when committed = 'N' then 0 else coalesce(undrawn_lim,0) end + coalesce(total_exp_usd,0) + coalesce(cont_out_bal,0)) end,0) as operational_exposure,
    coalesce(apportioned_value,0) as collateral_value_gross,
    coalesce(adj_coll_value,0) as collateral_value_adjusted,
    coalesce(greatest(case when status_code = 'NON-OPERATIONAL' then 0 else (case when committed = 'N' then 0 else coalesce(undrawn_lim,0) end + coalesce(total_exp_usd,0) + coalesce(cont_out_bal,0)) end - coalesce(adj_coll_value,0),0,0),0) as net_exposure,
    limit_classifier,
    oldfacility,
    restructured_facility,
    lorecqasmanager as lore_cqas_manager,
    baopmanager,
    marketing_flag,
    (select limit_category from lakehouse_prod.bronze_cbs.limit_liab_table where llt.limit_prefix||'/'||llt.limit_suffix = salesforce_facility_id) as limit_category,
    (select (select ref_desc from lakehouse_prod.bronze_cbs.reference_code_table where ref_rec_type = '4B' and ref_code = limit_category) from lakehouse_prod.bronze_cbs.limit_liab_table where llt.limit_prefix||'/'||llt.limit_suffix = salesforce_facility_id) as limit_category_desc
  from (
    select
      lim_exp_date,
      (select gh.group_id from lakehouse_prod.bronze_cbs.grouphousehold gh where gh.orgkey = fin_baop_001_ctr_b.cust_id) as group_id,
      (select groupname from lakehouse_prod.bronze_cbs.cifgroups where cifgroups.groupid = (select gh.group_id from lakehouse_prod.bronze_cbs.grouphousehold gh where gh.orgkey = fin_baop_001_ctr_b.cust_id) limit 1) as groupname,
      llt12.availability_end_date,
      llt12.lim_contract_date,
      (select main_dept from lakehouse_prod.silver_cbs.c_lnm_ext where c_lnm_ext.limit_prefix||'/'||c_lnm_ext.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id) as main_dept_code,
      (select (select ref_desc from lakehouse_prod.silver_cbs.c_lrct where ref_code = main_dept and ref_rec_type = 'LDEPT') from lakehouse_prod.silver_cbs.c_lnm_ext where c_lnm_ext.limit_prefix||'/'||c_lnm_ext.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id) as main_dept_code_desc,
      fin_baop_001_ctr_b.salesforce_facility_id,
      fin_baop_001_ctr_b.cust_name,
      fin_baop_001_ctr_b.limit_desc,
      fin_baop_001_ctr_b.risk_country_code,
      fin_baop_001_ctr_b.sales_sol_id,
      (select oau from lakehouse_prod.silver_cbs.cust_oau where countrycode = fin_baop_001_ctr_b.country_code) as oau_region,
      fin_baop_001_ctr_b.type_of_dept,
      fin_baop_001_ctr_b.trade_direct,
      fin_baop_001_ctr_b.primaryrmlogin_id,
      fin_baop_001_ctr_b.primary_dept_rm,
      fin_baop_001_ctr_b.lim_sanct_date,
      case when fin_baop_001_ctr_b.first_disb_dt is null then (select min_disbursement_date from lakehouse_prod.silver_cbs.ifrs9_min_disb_non_fund where parent_id = fin_baop_001_ctr_b.salesforce_facility_id) else fin_baop_001_ctr_b.first_disb_dt end as firstdisbursementdate,
      (select original_credit_grade from lakehouse_prod.silver_cbs.c_lnm_ext where c_lnm_ext.limit_prefix||'/'||c_lnm_ext.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id) as originalfacilitygrade,
      fin_baop_001_ctr_b.loan_grade,
      case 
        when fin_baop_001_ctr_b.loan_grade = '01' then 'VERY LOW RISK'
        when fin_baop_001_ctr_b.loan_grade in ('02','03') then 'LOW RISK'
        when fin_baop_001_ctr_b.loan_grade in ('04','05','06','07','08') then 'SATISFACTORY RISK'
        when fin_baop_001_ctr_b.loan_grade in ('09','10') then 'MODERATE RISK'
        when fin_baop_001_ctr_b.loan_grade = '11' then 'WATCH LIST RISK'
        when fin_baop_001_ctr_b.loan_grade = '12' then 'SUB-STANDARD RISK'
        when fin_baop_001_ctr_b.loan_grade = '13' then 'DOUBTFUL AND BAD DEBT RISK'
        when fin_baop_001_ctr_b.loan_grade = '14' then 'DEFAULT/LOSS RISK'
      end as riskcategory,
      fin_baop_001_ctr_b.status_code,
      fin_baop_001_ctr_b.orig_lim_exp_date,
      fin_baop_001_ctr_b.sector_being_financed,
      (select (select cl.localetext from lakehouse_prod.bronze_cbs.category_lang cl, lakehouse_prod.bronze_cbs.categories ct where ct.categoryid = cl.categoryid and ct.bank_id = cl.bank_id and ct.categorytype = 'SUB_SECTOR_CODE' and cl.localecode = 'en_US' and upper(ct.value) = sub_sector_being_financed and coalesce(ct.del_flg,'N') = 'N' and ct.bank_id = '01') from lakehouse_prod.silver_cbs.c_lnm_ext where limit_prefix||'/'||limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id) as subsector,
      (select subsectormap from lakehouse_prod.silver_cbs.cust_subsectormap where subsector = fin_baop_001_ctr_b.sub_sector_being_financed) as reportingsector,
      (select int_tbl_code from lakehouse_prod.silver_cbs.c_lnm_ext where c_lnm_ext.limit_prefix||'/'||c_lnm_ext.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id and bank_id = '01') as interesttablecode,
      (select acct_margin_pcnt from lakehouse_prod.silver_cbs.c_lnm_ext where c_lnm_ext.limit_prefix||'/'||c_lnm_ext.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id and bank_id = '01') as accountmargin,
      fin_baop_001_ctr_b.program_loan_type,
      lakehouse_prod.silver_cbs.get_new_currency_val(
        (select total_facility_amt from lakehouse_prod.silver_cbs.c_lnm_ext where c_lnm_ext.limit_prefix||'/'||c_lnm_ext.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id),
        (select total_facility_crncy from lakehouse_prod.silver_cbs.c_lnm_ext where c_lnm_ext.limit_prefix||'/'||c_lnm_ext.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id),
        'USD',
        lakehouse_prod.silver_cbs.get_bod_date('01')
      ) as totalfacilityamountusd,
      (select total_facility_amt from lakehouse_prod.silver_cbs.c_lnm_ext where c_lnm_ext.limit_prefix||'/'||c_lnm_ext.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id) as totalfacilityamtorgccy,
      (select total_facility_crncy from lakehouse_prod.silver_cbs.c_lnm_ext where c_lnm_ext.limit_prefix||'/'||c_lnm_ext.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id) as totalfacilitycurrency,
      (select (select ref_desc from lakehouse_prod.silver_cbs.c_lrct where ref_rec_type = 'LFACT' and ref_code = type_of_facility) from lakehouse_prod.silver_cbs.c_lnm_ext where c_lnm_ext.limit_prefix||'/'||c_lnm_ext.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id) as facilitytype,
      fin_baop_001_ctr_b.legalentity_type,
      case 
        when fin_baop_001_ctr_b.legalentity_type in ('LIMITED LIABILITY COMPANY','MULTILATERAL','NON - GOVERNMENT FINANCIAL INSTITUTION','PRIVATE COMPANY LIMITED BY SHARES (LTD)','PUBLIC LIMITED COMPANY (PLC)') then 'PRIVATE'
        when fin_baop_001_ctr_b.legalentity_type in ('GOVERNMENT FINANCIAL INSTITUTION') then 'PUBLIC'
        when fin_baop_001_ctr_b.legalentity_type in ('MINISTRY', 'PARASTATAL','CENTRAL BANK') then 'SOVEREIGN'
      end as borrowertype,
      fin_baop_001_ctr_b.orig_sanct_lim,
      lakehouse_prod.silver_cbs.get_new_currency_val(
        fin_baop_001_ctr_b.orig_sanct_lim,
        'USD',
        (select llt.crncy_code from lakehouse_prod.bronze_cbs.limit_liab_table where llt.limit_prefix||'/'||llt.limit_suffix = salesforce_facility_id),
        lakehouse_prod.silver_cbs.get_bod_date('01')
      ) as approvedlimitorgccy,
      (select committed_flg from lakehouse_prod.bronze_cbs.limit_liab_table where llt.limit_prefix||'/'||llt.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id) as committed,
      fin_baop_001_ctr_b.undrawn_lim,
      lakehouse_prod.silver_cbs.get_new_currency_val(
        undrawn_lim,
        'USD',
        (select crncy_code from lakehouse_prod.bronze_cbs.limit_liab_table where llt.limit_prefix||'/'||llt.limit_suffix = salesforce_facility_id),
        lakehouse_prod.silver_cbs.get_bod_date('01')
      ) as undrawnlimitorgccy,
      fin_baop_001_ctr_b.funded_out_bal,
      vloanfac.total_exp_usd,
      fin_baop_001_ctr_b.funded_out_bal_org,
      case when llt12.crncy_code = 'USD' then vloanfac.total_exp_usd else coalesce(lakehouse_prod.silver_cbs.get_new_currency_val(total_exp_usd,'USD',llt12.crncy_code,(select date_sub(db_stat_date, 1) from lakehouse_prod.bronze_cbs.sol_group_control_table)),0) end as total_exp,
      fin_baop_001_ctr_b.cont_out_bal - coalesce((select contingentliabamtinusd1 from lakehouse_prod.silver_cbs.view_fac_master_gcaf where fin_baop_001_ctr_b.salesforce_facility_id = parent_id limit 1),0) as cont_out_bal,
      fin_baop_001_ctr_b.cont_out_bal_org - coalesce((select contingentliabamt1 from lakehouse_prod.silver_cbs.view_fac_master_gcaf where fin_baop_001_ctr_b.salesforce_facility_id = parent_id limit 1),0) as cont_out_bal_org,
      fin_baop_001_ctr_b.gross_exp,
      fin_baop_001_ctr_b.oper_exposure,
      fin_baop_001_ctr_b.apportioned_value,
      fin_baop_001_ctr_b.adj_coll_value,
      case when sign(fin_baop_001_ctr_b.net_exposure) = -1 then 0 else fin_baop_001_ctr_b.net_exposure end as netexposureaftermitiga,
      (select case when ln_bgen.limit_classifier = 'B' then 'Bilateral' when ln_bgen.limit_classifier = 'A' then 'Agent and participant' when ln_bgen.limit_classifier = 'S' then 'Syndicated Participation' else 'Club Deal' end from lakehouse_prod.bronze_cbs.ln_general_details_table, lakehouse_prod.bronze_cbs.limit_liab_table where ln_bgen.limit_b2kid = llt.limit_b2kid and llt.limit_prefix||'/'||llt.limit_suffix = salesforce_facility_id) as limit_classifier,
      (select limit_ctrl_ctr from lakehouse_prod.bronze_cbs.ln_general_details_table, lakehouse_prod.bronze_cbs.limit_liab_table where ln_bgen.limit_b2kid = llt.limit_b2kid and llt.limit_prefix||'/'||llt.limit_suffix = salesforce_facility_id) as oldfacility,
      (select restructured_facility from lakehouse_prod.silver_cbs.c_lnm_ext where c_lnm_ext.limit_prefix||'/'||c_lnm_ext.limit_suffix = fin_baop_001_ctr_b.salesforce_facility_id) as restructured_facility,
      (select emp_name from lakehouse_prod.bronze_cbs.gen_emp_table where emp_id = (select tertiaryrmlogin_id from lakehouse_prod.bronze_cbs.corporate where corp_key = fin_baop_001_ctr_b.cust_id)) as lorecqasmanager,
      (select emp_name from lakehouse_prod.bronze_cbs.gen_emp_table where emp_id = (select secondrmlogin_id from lakehouse_prod.bronze_cbs.corporate where corp_key = fin_baop_001_ctr_b.cust_id)) as baopmanager,
      llt12.modify_delete_reason_code,
      (select ref_desc from lakehouse_prod.bronze_cbs.reference_code_table where ref_rec_type = 'IT' and ref_code = llt12.modify_delete_reason_code) as marketing_flag
    from lakehouse_prod.silver_cbs.main_view_facility_mast fin_baop_001_ctr_b,
         lakehouse_prod.bronze_cbs.limit_liab_table llt12
         left join lakehouse_prod.silver_cbs.facility_mast_loan_exp_view vloanfac on vloanfac.parent_limit = fin_baop_001_ctr_b.salesforce_facility_id
    where fin_baop_001_ctr_b.salesforce_facility_id = llt12.limit_prefix || '/' || llt12.limit_suffix
      and llt12.del_flg != 'Y'
      and limit_state != 'C'
      and not exists (select 1 from lakehouse_prod.silver_cbs.view_cot_chrge_off_new cot where cot.parent_id = fin_baop_001_ctr_b.salesforce_facility_id)
  ) 
  where gross_exposure != 0
)